## Code to Chapter 6 of LangChain for Life Science and Healthcare book, by Dr. Ivan Reznikov

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare)

> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

# Finding Typos in PubChem Database

This notebook demonstrates how to use LangChain and OpenAI's GPT models to generate potential typos for chemical terms and then search the PubChem database to see how many compounds are associated with these typos. This approach can help identify common misspellings in chemical databases and potentially find compounds that were indexed under incorrect spellings.

## Overview

The workflow consists of three main steps:
1. **Setup**: Install required packages and configure API keys
2. **Typo Generation**: Use LangChain with OpenAI's GPT model to generate potential typos for a given chemical term
3. **Database Search**: Query PubChem's API to find how many compounds are associated with each potential typo


## Environment Setup and Package Installation

First, we need to install the required packages for LangChain integration with OpenAI.

In [1]:
# @title Installing Python dependencies
%pip install -q "rdkit==2023.9.6" "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4.0" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.8/34.8 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source 

In [2]:
# @title Verifying versions of Python dependencies
!pip freeze | grep "lang\|openai"

google-ai-generativelanguage==0.6.15
google-cloud-language==2.21.0
langchain==1.0.0
langchain-classic==1.0.3
langchain-community==0.4
langchain-core==1.2.30
langchain-openai==1.0.0
langchain-protocol==0.0.18
langchain-text-splitters==1.1.1
langgraph==1.0.10
langgraph-checkpoint==4.1.1
langgraph-prebuilt==1.0.10
langgraph-sdk==0.3.15
langsmith==0.10.2
libclang==18.1.1
openai==2.45.0


In [3]:
# @title Setting environmental variables
import os

# --- Dual-mode secrets: works in Google Colab AND locally (.env / environment) ---
try:
    from google.colab import userdata  # type: ignore

    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if not IN_COLAB:
    # Local run: load variables from a .env file if present (never commit .env!).
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except Exception:
        pass


def get_secret(name, default=None):
    """Read a secret from Colab Secrets, else from local env/.env, else default."""
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)


# 👇 Choose your provider 👇
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"

if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret(
        "LC4LSH_ANTHROPIC_API_KEY", "sk-ant-..."
    )
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")

print(
    f"✅ API keys loaded for {API_KEY_PROVIDER} (source: {'Colab Secrets' if IN_COLAB else 'local env/.env'})"
)

# Hugging Face token (optional; needed for gated models)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""

✅ API keys loaded for OPENAI (source: Colab Secrets)


In [4]:
# @title Setting LangSmith variables
LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "") or get_secret(
    "LANGSMITH_API_KEY", ""
)
LANGSMITH_PROJECT = "lc4lsh-chapter6-pubchem-typos"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")

LangSmith ON -> lc4lsh-chapter6-pubchem-typos


## LangChain Setup for Typo Generation

Here we create a LangChain pipeline that uses OpenAI's GPT model to generate potential typos.


In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [6]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a professional editor and typo-catcher",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

typo_chain = prompt | llm | StrOutputParser()

**Chain Explanation**:
- The `|` operator creates a processing pipeline
- Input flows through: prompt formatting → LLM processing → string parsing
- This is LangChain's "Expression Language" for building data processing chains

## PubChem API Integration

We'll create a function to query PubChem's database for compound information.


In [7]:
import requests


def get_pubchem_data(subword):
    response = requests.get(
        "https://pubchem.ncbi.nlm.nih.gov/sdq/sdqagent.cgi?infmt=json&outfmt=json&query={%22select%22:%22*%22,%22collection%22:%22compound%22,%22order%22:[%22relevancescore,desc%22],%22start%22:1,%22limit%22:10,%22where%22:{%22ands%22:[{%22*%22:%22"
        + subword
        + "%22}]},%22width%22:1000000,%22listids%22:0}",
        cookies={},
        headers={},
    )
    return response

**API URL Breakdown**:
- `infmt=json&outfmt=json`: Input and output format specification
- `select=*`: Select all available fields
- `collection=compound`: Search in the compound database
- `order=[relevancescore,desc]`: Sort by relevance (highest first)
- `limit=10`: Return maximum 10 results
- `where.ands[0]["*"]`: Search for the subword in any field

## Generate and Test Typos

Now we'll generate potential typos for a chemical term and search PubChem for each one.

In [8]:
word = "ethyl"
typo_llm_response = typo_chain.invoke(f"""
  Return a semicolon-separated list of 10 most possible typos for word {word}.
  The resposnse should contain only possible typos!
  Don't include initial word {word}
  Avoid adding duplicates.""")
typo_list = [x.strip() for x in typo_llm_response.replace(".", "").split(";")]

In [9]:
typo_list, len(set(typo_list))

(['ehtyl',
  'etyl',
  'ethly',
  'eythl',
  'ehtly',
  'etyhl',
  'ethli',
  'eylth',
  'ethly',
  'ehylt'],
 9)

## Search PubChem for Each Typo

Finally, we'll search PubChem for each generated typo to see how many compounds are associated with it.


In [10]:
typo_dict = {}
for subword in typo_list:
    response = get_pubchem_data(subword)
    total_count = response.json()["SDQOutputSet"][0]["totalCount"]
    if total_count:
        typo_dict[subword] = total_count

In [11]:
typo_dict

{'ehtyl': 53, 'etyl': 29, 'ethly': 2847, 'ehtly': 9, 'etyhl': 1, 'ethli': 2847}

## Results Analysis and Interpretation

The results will show which "typos" actually correspond to real compounds in PubChem. This can reveal:

**Most Common Typos Found:**
- **'ethly'** (2,453 occurrences) - This is by far the most frequent typo, where the 'y' and 'l' are swapped. This makes sense because it's a simple transposition error that's easy to make when typing quickly.

**Moderately Common Typos:**
- **'ehtyl'** (53 occurrences) - Here the 'e' and 'h' are swapped, another transposition error
- **'etyl'** (23 occurrences) - Missing the 'h' entirely, likely from fast typing or autocorrect issues

**Rare Typos:**
- **'ehtly'** (10 occurrences) - Combines both letter swapping (e/h) and omission (missing 'y')
- **'etyhl'** (1 occurrence) - Multiple letter rearrangements

**What This Reveals:**
1. **Real-world data quality issues**: Even in scientific databases like PubChem, human input errors occur and persist
2. **Common error patterns**: Transposition errors (swapping adjacent letters) are the most frequent type of typo
3. **Impact magnitude**: The 'ethly' typo appears in nearly 2,500 compound entries, suggesting this is a systematic issue that could affect chemical literature searches and data retrieval

### Potential Applications:

- **Database Curation**: Identify potential spelling errors in chemical databases
- **Search Enhancement**: Improve chemical search engines with common misspelling patterns
- **Quality Control**: Validate chemical nomenclature in research databases


## Limitations & safety notes

- LLM-generated 'typos' are synthetic perturbations for testing identifier robustness, not real database errors.
- **Paid OpenAI API required**.
- PubChem lookups are rate-limited; cache results.


In [12]:
# Cleanup
import gc

for _v in (
    "llm",
    "model",
    "agent",
    "agent_executor",
    "graph",
    "app",
    "workflow",
    "agent_run",
):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")

Cleanup complete.


## Exercises

<details><summary>Why test robustness to name typos?</summary>Real queries contain misspellings; a robust pipeline must still resolve the intended compound identifier.</details>

<details><summary>Why resolve to an exact identifier?</summary>InChIKey/CID are unambiguous, unlike free-text names that vary with typos/synonyms.</details>

<details><summary>Why cache PubChem results?</summary>To avoid rate limits and make runs reproducible.</details>

### Tasks
- **Task A** - Add fuzzy matching to recover the correct compound from a misspelled name.
- **Task B** - Cache PubChem lookups to a local JSON and reload between runs.
- **Task C** - Measure resolution accuracy over a small typo set.
- **Task D** - Add an abstention path when no confident identifier match exists.
